In [26]:
METHOD = 'ghmm'
DATA_PATH = "/home/denklewer/Data/"
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/'+ METHOD+'/'
DECOY_KEY = 'x1'


# 0. Configure logging (just for convinience)

In [1]:
import os, sys
import pandas as pd
import numpy as np
import time
import gzip
import mhcnames
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import ghmm

# log library versions and working dir
from scipy.version import full_version as spversion

In [2]:
import os
import logging
import sys
from __future__ import print_function
LOG_DIR = 'logs/'

if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)



logging.basicConfig(
    filename='{}{}_train.log'.format(LOG_DIR,METHOD), 
    filemode='a',
    encoding='utf-8', 
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)-1s: %(message)s',
            datefmt='%d-%m-%y %H:%M:%S') 

# console = logging.StreamHandler()
# console.setLevel(logging.INFO)
# formatter = logging.Formatter(fmt='[%(asctime)s] %(levelname)-1s: %(message)s',datefmt='%d-%m-%y %H:%M:%S' )
# # tell the handler to use this format
# console.setFormatter(formatter)
# # add the handler to the root logger
# logging.getLogger('').addHandler(console)

logger = logging.getLogger("GHMM")
logger.setLevel(logging.INFO)

# 1. Check Packages

In [3]:
logger.info('Python    :{}'.format(sys.version))
logger.info('Numpy     :{}'.format(np.__version__))
logger.info("Scipy     :{}".format(spversion))
logger.info('Pandas    :{}'.format(pd.__version__))

logger.info("GHMM:      0.9 (manually tracked)") # no such thing as ghmm.__version__, and not under pip
logger.info('Workdir   :{}'.format(os.getcwd()))

GHMM <ipython-input-3-6472529ea058>:1 - Python    :2.7.18 |Anaconda, Inc.| (default, Jun  4 2021, 14:47:46) 
[GCC 7.3.0]
GHMM <ipython-input-3-6472529ea058>:2 - Numpy     :1.16.6
GHMM <ipython-input-3-6472529ea058>:3 - Scipy     :1.2.3
GHMM <ipython-input-3-6472529ea058>:4 - Pandas    :0.24.2
GHMM <ipython-input-3-6472529ea058>:6 - GHMM:      0.9 (manually tracked)
GHMM <ipython-input-3-6472529ea058>:7 - Workdir   :/home/denklewer/Projects/MHC_predictor


# 2. Read data

In [4]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method

ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

In [5]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [7]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)


/home/denklewer/Tools/anacoda3/envs/ghmm/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2818: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


In [8]:
#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides
def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()
per_allele_binders_train = dict()
per_allele_non_binders_train = dict()

for allele_name in ALLELES:
    df_train = per_allele_train_df[allele_name]
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)

addAuxColumns(per_allele_train_df)    
addAuxColumns(per_allele_val_df)   

# 3. Training code

In [10]:
#functions for training

# define the amino acids alphabet with the stop emission
AminoAcidsZ = ghmm.Alphabet(ghmm.AminoAcids.listOfCharacters + ['Z'])
def randomInitHMMwithSpecialStates(numStates, alphabet):
    """Initialize an HMM

    Create an HMM with a random state and dedicated stop state and emission
    numStates - number of hidden states in the HMM
    alphabet - the alphabet of emissions, of ghmm.Alphabet type,
    should contain X and Z emissions for start and stop
    """
    
    # all states equiprobable as initial, Baum-Welch will adjust
    pi = [1] * numStates
    
    # transition probabilities: all random except for the last state, which does not map into anything,
    # so ghmm will recognize it as the final state
    tT = np.random.rand(numStates, numStates)
    tT[numStates - 1] = np.array([0] * numStates)
    
    # emission probabilities: last state will be emitting only the dedicated stop state 'Z'
    # other options not possible:
    # - if we does not let the last state emit at all (silent state), ghmm.baumWelch
    #   will not work as it does not support silent states
    # - if we leave the random emission pattern for the last state, baumWelch complains
    #   that the state cannot be reached (And so the model is wrong)
    # so we need the dedicated stop state
    # Same for the start state
    tE = np.random.rand(numStates, len(alphabet))
    tE[numStates - 1] = np.zeros(len(AminoAcidsZ))
    tE[numStates - 1][AminoAcidsZ.internal('Z')] = 1
    for i in range(0, numStates - 1):
        tE[i][AminoAcidsZ.internal('Z')] = 0
    
    h = ghmm.HMMFromMatrices(alphabet, ghmm.DiscreteDistribution(alphabet), tT, tE, pi)
    h.normalize()
    return h

def trainHMMs(binders, nonbinders, numStatesBinders, numStatesNonbinders, llCutoff):
    
    logger.info(
        'Training HMM on peptides ({} B, {} NB)'.format(
            len(binders),
            len(nonbinders)
        )
    )
    
    # convert peptides to ghmm structure
    bindersTrainSS = ghmm.SequenceSet(
        AminoAcidsZ,
        [list(str(p) + 'Z') for p in binders.peptide])
    hmmB = randomInitHMMwithSpecialStates(numStatesBinders, AminoAcidsZ)
    tt0 = time.clock()
    hmmB.baumWelch(bindersTrainSS, loglikelihoodCutoff = llCutoff)
    tt1 = time.clock()
    logger.info('Binders model trained in {} s'.format(tt1 - tt0))
    
    nonbindersTrainSS = ghmm.SequenceSet(
        AminoAcidsZ,
        [list(str(p) + 'Z') for p in nonbinders.peptide])
    hmmN = randomInitHMMwithSpecialStates(numStatesNonbinders, AminoAcidsZ)
    tt0 = time.clock()
    hmmN.baumWelch(nonbindersTrainSS, loglikelihoodCutoff = llCutoff)
    tt1 = time.clock()
    logger.info('Non-binders model trained in {} s'.format(tt1 - tt0))
    
    return (hmmB, hmmN)

def prepareHmmSequenceSets(peptides, stopEmission=True):
    
    t0 = time.clock()
    
    if stopEmission:
        peptideList = [str(p) + 'Z' for p in peptides]
    else:
        peptideList = [str(p) for p in peptides]
    
    # need to split in blocks in case there are more than 1.5M peptides (ghmm limitation)
    ghmmMaxSequencesPerSet = 1500000
    lenPeptides = len(peptides)
    numFullBlocks = lenPeptides // ghmmMaxSequencesPerSet
    lenLastBlock = lenPeptides % ghmmMaxSequencesPerSet
    #print('Total peptides:', lenPeptides)
    #print('Full blocks:', numFullBlocks)
    #print('Peptides in the last block:', lenLastBlock)
    
    sequenceSets = []
    # full blocks (if any)
    for i in range(0, numFullBlocks):
        rangeStart = i * ghmmMaxSequencesPerSet
        rangeEnd = rangeStart + ghmmMaxSequencesPerSet
        #print('Converting block {}, range {}:{}...'.format(i, rangeStart, rangeEnd))
        sequenceBlock = ghmm.SequenceSet(
            AminoAcidsZ,
            [list(p) for p in peptideList[rangeStart:rangeEnd]]
        )
        sequenceSets.append(sequenceBlock)
    # last block
    rangeStart = numFullBlocks * ghmmMaxSequencesPerSet
    rangeEnd = lenPeptides
    if (lenLastBlock > 0):
        #print('Converting last block, range {}:{}...'.format(rangeStart, rangeEnd))
        sequenceBlock = ghmm.SequenceSet(
                AminoAcidsZ,
                [list(p) for p in peptideList[rangeStart:rangeEnd]]
            )
        sequenceSets.append(sequenceBlock)
    #print('Total blocks:', len(sequenceSets))
    t1 = time.clock()
    #print('Splitting and conversion to ghmm structures done in {} s'.format(t1 - t0))
    
    return sequenceSets

def predictHmmPeptidesRaw(hmmModelFile, sequenceSets):
    """Predict -10logp values using a given HMM model for binders and non-binders.

       Results will contain -inf values """

    hmm = ghmm.HMMOpen(hmmModelFile)

    return np.concatenate(
        [-10 * np.array(hmm.loglikelihoods(s)) for s in sequenceSets]
    )

In [12]:
from sklearn.model_selection import KFold
for df in per_allele_train_df.values():
    print("max length {}".format(max(df.peptide.str.len())))
kf = KFold(n_splits=5)
llCutoff = 0.00005

max length 14
max length 30
max length 14
max length 30
max length 14
max length 14
max length 30


In [11]:
for allele_name in ALLELES:
    print(allele_name)
    if defineClass(allele_name) == "I":
        numStatesBinders = 25
        numStatesNonbinders = 25
    else:
        numStatesBinders = 40
        numStatesNonbinders = 25

        
    binders_array = per_allele_binders_train[allele_name]
    non_binders_array = per_allele_non_binders_train[allele_name]
    i = 0
    for train_indexes_b, test_indexes_b in kf.split(binders_array):
        i = i + 1
        logger.info("split {}".format(i))
        current_train_b = binders_array[train_indexes_b]
        current_test_b = binders_array[test_indexes_b]
        current_train_nb = non_binders_array
        current_test_nb = non_binders_array
    
        df_binders = pd.DataFrame({"peptide":current_train_b})
        df_non_binders = pd.DataFrame({"peptide":current_train_nb})
        hmmB, hmmN = trainHMMs(df_binders,df_non_binders, numStatesBinders, numStatesNonbinders, llCutoff)
        hmmB.write('trained_models/{}-binders-{}.xml'.format(allele_name, i))
        hmmN.write('trained_models/{}-nonbinders-{}.xml'.format(allele_name, i))



GHMM <ipython-input-11-2e137067deae>:23 - split 1
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (4960 B, 6200 NB)


HLA_A_02_01


GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 83 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 67.008256 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 108 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 123.079587 s
GHMM <ipython-input-11-2e137067deae>:23 - split 2
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (4960 B, 6200 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 77 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 62.598569 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 103 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 115.850396 s
GHMM <ipython-input-11-2e137067deae>:23 - split 3
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (4960 B, 6200 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(8

HLA_B_27_05


GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 109 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 222.803187 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 109 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 302.702345 s
GHMM <ipython-input-11-2e137067deae>:23 - split 2
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (12168 B, 15210 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 113 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 234.315303 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 109 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 302.799454 s
GHMM <ipython-input-11-2e137067deae>:23 - split 3
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (12168 B, 15210 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum

HLA_A_03_01


... (5 outputs left out)

HLA_DRB1_04_01


GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 215 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 416.703382 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 3 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 3.039507 s
GHMM <ipython-input-11-2e137067deae>:23 - split 2
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (2396 B, 3017 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 186 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 355.63861 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 3 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 3.277373 s
GHMM <ipython-input-11-2e137067deae>:23 - split 3
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (2396 B, 3017 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): 

HLA_DRB1_04_05


GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 131 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 120.120262 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 3 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 1.652779 s
GHMM <ipython-input-11-2e137067deae>:23 - split 2
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (1342 B, 1677 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 138 steps
GHMM <ipython-input-9-bea497add1a6>:57 - Binders model trained in 130.57622 s
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): Convergence after 3 steps
GHMM <ipython-input-9-bea497add1a6>:66 - Non-binders model trained in 1.648947 s
GHMM <ipython-input-11-2e137067deae>:23 - split 3
GHMM <ipython-input-9-bea497add1a6>:45 - Training HMM on peptides (1342 B, 1677 NB)
GHMM ghmm.py:148 - reestimate.c:ghmm_dmodel_baum_welch(853): 

# 4. Prediction code

In [13]:
logger.setLevel(logging.CRITICAL)

In [24]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    print(allele_name)
    print('Computing HMM predictions')
    df_val = per_allele_val_df[allele_name][['peptide', 'length', 'binder']].copy()
    df_val[RESULT_COLUMNS['id']] = 'seq'  + df_val.index.astype(str).values

    
    sequenceSet = prepareHmmSequenceSets(df_val.peptide)
    
    # make raw predictions (-10logp)
    for i in range(1, kf.n_splits + 1):
        print("Predict with model-{}".format(i))
        print("Binders")
        df_val['HMMB_{}'.format(i)] = predictHmmPeptidesRaw(
            'trained_models/{}-binders-{}.xml'.format(allele_name, i),
            sequenceSet
        )
        print("Non-Binders")
        df_val['HMMN_{}'.format(i)] = predictHmmPeptidesRaw(
            'trained_models/{}-nonbinders-{}.xml'.format(allele_name, i),
            sequenceSet
        )
        
    # normalize by length, replace infs by a large value, and take deltas
    columnsHMMB = ['HMMB_' + str(i) for i in range(1, kf.n_splits + 1)]
    columnsHMMN =  ['HMMN_' + str(i) for i in range(1, kf.n_splits + 1)]
    columnsHMMD = ['HMMD_{}'.format(i) for i in range(1, kf.n_splits + 1)]
    columnsHMM = columnsHMMB + columnsHMMN
    
    df_val.loc[:, columnsHMM] = df_val.loc[:, columnsHMM].div(df_val.length, axis=0)
    for i, (b_column, n_column, d_column) in enumerate(zip(columnsHMMB, columnsHMMN, columnsHMMD)):
        df_val[d_column] = df_val[b_column] - df_val[n_column]
        
    df_val = df_val.replace(np.inf, 10000.0)
    df_val['HMMB'] = df_val[columnsHMMB].median(axis=1)  
    df_val['HMMN'] = df_val[columnsHMMN].median(axis=1)
    df_val['HMMD'] = df_val[columnsHMMD].median(axis=1)
    # merge with orginal dataframe
    per_allele_predictions[allele_name] = df_val

HLA_A_02_01
Computing HMM predictions
Predict with model-1
Binders
Non-Binders
Predict with model-2
Binders
Non-Binders
Predict with model-3
Binders
Non-Binders
Predict with model-4
Binders
Non-Binders
Predict with model-5
Binders
Non-Binders
HLA_B_27_05
Computing HMM predictions
Predict with model-1
Binders
Non-Binders
Predict with model-2
Binders
Non-Binders
Predict with model-3
Binders
Non-Binders
Predict with model-4
Binders
Non-Binders
Predict with model-5
Binders
Non-Binders
HLA_A_03_01
Computing HMM predictions
Predict with model-1
Binders
Non-Binders
Predict with model-2
Binders
Non-Binders
Predict with model-3
Binders
Non-Binders
Predict with model-4
Binders
Non-Binders
Predict with model-5
Binders
Non-Binders
HLA_B_07_02
Computing HMM predictions
Predict with model-1
Binders
Non-Binders
Predict with model-2
Binders
Non-Binders
Predict with model-3
Binders
Non-Binders
Predict with model-4
Binders
Non-Binders
Predict with model-5
Binders
Non-Binders
HLA_DRB1_01_01
Computing HMM

## Write to files

In [30]:
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)
for allele_name, df in per_allele_predictions.items():
    df.to_csv("{}ghmm_predictions_{}".format(TARGET_PATH_TO_PREDICTIONS,allele_name))